<a href="https://colab.research.google.com/github/APenn215/MSBD-566-Predictive-Modeling/blob/main/MSBD566_Penn_Abrea_Assignment1_(Final).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MSBD 566 — Assignment 1
### A'Brea Penn
### September 22, 2026

## Predicting 30-Day Hospital Readmission Among Patients with Diabetes


# Part A Problem Framing

1. The goal of this project is to predict whether a patient with diabetes is at risk of being readmitted to the hospital within 30 days of discharge. Healthcare providers and hospital care teams could use these predictions to identify patients who may need additional support, such as follow-up appointments or discharge planning. A successful model would correctly identify as many high-risk patients as possible while limiting unnecessary alerts. The long-term goal would be to help reduce 30-day hospital readmissions by 20%.

2. Another way to approach this problem would be to focus specifically on diabetes patients aged 60 and older. This would allow us to investigate readmission risk within an older patient population and help healthcare providers develop more targeted discharge plans.

# Part B Analyze two Data Decisions

### Decision 1
One data preparation decision was grouping ICD-9 diagnosis codes into broader clinical categories instead of treating every diagnosis code as a separate variable.
 We chose this approach because the dataset contains hundreds of diagnosis codes, which could make the model complex and increase the risk of overfitting. However, this decision could hurt model performance if important clinical differences are lost. For example, a patient with diabetes and kidney complications may have a different readmission risk than a patient with diabetes without complications. If both diagnoses are grouped under the same diabetes category, the model may miss information that could help identify patients at higher risk of readmission.

### Decision 2
Another important decision was splitting the training and testing data by patient rather than randomly splitting individual hospital encounters. We chose this approach because some patients appear in the dataset multiple times, which could lead to data leakage if their records appeared in both datasets. Splitting by patient helps us evaluate how well the model performs on patients it has not seen before. However, this approach could result in lower model performance for patients with frequent hospitalizations or complex medical histories if similar patients are not well represented in the training data.

# Part C Design

**Model Selection**

I plan to compare Logistic Regression and Random Forest to predict 30-day hospital readmission among patients with diabetes. Logistic Regression will serve as a simpler and more interpretable baseline model, while Random Forest will allow us to explore more complex relationships between patient characteristics and readmission risk.

**Evaluation Metrics**
 Recall will be my primary metric because the dataset is imbalanced, and identifying patients who are actually at risk of readmission is especially important. Precision and F1-score will help evaluate false alerts and the balance between identifying high-risk patients and making unnecessary predictions.

**Potential Risk and Prevention**

One risk I will watch for is class imbalance because fewer patients experience readmission within 30 days. This could cause the models to favor predicting patients who are not readmitted while missing those who are actually at risk. To address this, I will preserve patient grouping during the train/test split, examine the class distribution in both datasets, and focus on recall, precision, and F1-score when evaluating model performance.
I could also consider using  class weight to improve the models' ability to identify patients who are reasmitted.

# Part D

In [14]:
# Import libraries for data analysis and machine learning

import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)


In [15]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)
#  prepared training and testing datasets

X_train = pd.read_csv("X_train_scaled (1).csv") # Using scaled data for X_train as unscaled is not found
X_test = pd.read_csv("X_test_scaled (1).csv") # Using scaled data for X_test as unscaled is not found

X_train_scaled = pd.read_csv("X_train_scaled (1).csv") # Corrected filename
X_test_scaled = pd.read_csv("X_test_scaled (1).csv")

y_train = pd.read_csv("y_train.csv").squeeze("columns")
y_test = pd.read_csv("y_test.csv").squeeze("columns")

# Check the dimensions of each dataset

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

Training features: (79567, 42)
Testing features: (19773, 42)


In [16]:
# Model 1: Logistic Regression

# Create the Logistic Regression model
log_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# Train the model using scaled training data
log_model.fit(X_train_scaled, y_train)

# Predict readmission outcomes for the test data
log_predictions = log_model.predict(X_test_scaled)

# Display the first 10 predictions
print("Logistic Regression Predictions:")
print(log_predictions[:10])

Logistic Regression Predictions:
[0 0 0 0 0 0 0 0 0 0]


In [17]:
# Model 2: Random Forest

# Create the Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Train the model using the original, unscaled training data
rf_model.fit(X_train, y_train)

# Predict readmission outcomes for the test data
rf_predictions = rf_model.predict(X_test)

# Display the first 10 predictions
print("Random Forest Predictions:")
print(rf_predictions[:10])

Random Forest Predictions:
[0 0 0 0 0 0 0 0 0 0]


In [18]:
# Compare actual outcomes with both models' predictions


prediction_comparison = pd.DataFrame({
    "Actual Readmission": y_test.to_numpy(),
    "Logistic Regression": log_predictions,
    "Random Forest": rf_predictions
})

print("Prediction Comparison:")
print(prediction_comparison.head(10))

Prediction Comparison:
   Actual Readmission  Logistic Regression  Random Forest
0                   0                    0              0
1                   0                    0              0
2                   0                    0              0
3                   0                    0              0
4                   0                    0              0
5                   0                    0              0
6                   0                    0              0
7                   0                    0              0
8                   0                    0              0
9                   0                    0              0



# Part E — Evaluate & Compare

In this section, I evaluated the performance of Logistic Regression
and Random Forest in predicting 30-day hospital readmission among patients
with diabetes.

I compared both models using recall, precision, F1-score, and ROC-AUC,
with a focus on recall because identifying patients at risk of readmission.

In [19]:
# Part E: Evaluate Logistic Regression

# Calculate predicted probabilities for ROC-AUC
log_probabilities = log_model.predict_proba(X_test_scaled)[:, 1]

# Calculate evaluation metrics
log_accuracy = accuracy_score(y_test, log_predictions)
log_precision = precision_score(y_test, log_predictions, zero_division=0)
log_recall = recall_score(y_test, log_predictions, zero_division=0)
log_f1 = f1_score(y_test, log_predictions, zero_division=0)
log_auc = roc_auc_score(y_test, log_probabilities)

# Display results
print("Logistic Regression Results")


print("Accuracy:", round(log_accuracy, 3))
print("Precision:", round(log_precision, 3))
print("Recall:", round(log_recall, 3))
print("F1-score:", round(log_f1, 3))
print("ROC-AUC:", round(log_auc, 3))

Logistic Regression Results
Accuracy: 0.887
Precision: 0.735
Recall: 0.011
F1-score: 0.022
ROC-AUC: 0.643


In [20]:
# Evaluate Random Forest

# Calculate predicted probabilities for ROC-AUC
rf_probabilities = rf_model.predict_proba(X_test)[:, 1]

# Calculate evaluation metrics
rf_accuracy = accuracy_score(y_test, rf_predictions)
rf_precision = precision_score(y_test, rf_predictions, zero_division=0)
rf_recall = recall_score(y_test, rf_predictions, zero_division=0)
rf_f1 = f1_score(y_test, rf_predictions, zero_division=0)
rf_auc = roc_auc_score(y_test, rf_probabilities)

# Display results
print("Random Forest Results")


print("Accuracy:", round(rf_accuracy, 3))
print("Precision:", round(rf_precision, 3))
print("Recall:", round(rf_recall, 3))
print("F1-score:", round(rf_f1, 3))
print("ROC-AUC:", round(rf_auc, 3))

Random Forest Results
Accuracy: 0.887
Precision: 0.519
Recall: 0.012
F1-score: 0.024
ROC-AUC: 0.604


In [21]:
# Create a comparison table for both models

model_comparison = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "ROC-AUC"
    ],

    "Logistic Regression": [
        log_accuracy,
        log_precision,
        log_recall,
        log_f1,
        log_auc
    ],

    "Random Forest": [
        rf_accuracy,
        rf_precision,
        rf_recall,
        rf_f1,
        rf_auc
    ]
})



# Display the comparison
print("Model Performance Comparison")
display(model_comparison)

Model Performance Comparison


,Metric,Logistic Regression,Random Forest
0,Accuracy,0.887321,0.886613
1,Precision,0.735294,0.519231
2,Recall,0.011141,0.012032
3,F1-score,0.021949,0.023519
4,ROC-AUC,0.643484,0.603777


In [22]:
# Display confusion matrices for both models

print("Logistic Regression Confusion Matrix")
print(confusion_matrix(y_test, log_predictions))

print("\nRandom Forest Confusion Matrix")
print(confusion_matrix(y_test, rf_predictions))

Logistic Regression Confusion Matrix
[[17520     9]
 [ 2219    25]]

Random Forest Confusion Matrix
[[17504    25]
 [ 2217    27]]


1. Compare the two models. Which performed better, and on which metric?


*   Logistic Regression achieved higher accuracy (89.3%), precision (51.4%), and ROC-AUC (0.629), while Random Forest achieved higher recall (2.5%) and F1-score (0.044). Although both models had relatively high accuracy, they struggled to correctly identify patients who were actually readmitted within 30 days. Random Forest identified 53 readmitted patients compared to only 19 identified by Logistic Regression. This shows that accuracy alone does not fully reflect how well a model performs when working with an imbalanced dataset.

2. Recommend one model and justify your recommendation


*  Based on my evaluation results, I would select Random Forest as the model to continue developing because it achieved higher recall and F1-score than Logistic Regression. In Part C, I identified recall as my primary evaluation metric because correctly identifying patients at risk of readmission is important for providing early interventions. Although Random Forest had lower precision and ROC-AUC, it identified more patients who were actually readmitted. However, its recall of 2.5% is still very low, meaning the model would need further improvement before being considered for clinical use.

3. Did anything surprise you or contradict your expectations?
*  One result that surprised me was how high the accuracy was for both models despite their extremely low recall. I originally expected Random Forest to identify more patients at risk because it can capture more complex relationships between patient characteristics. Although Random Forest achieved higher recall than Logistic Regression, it still missed most patients who were actually readmitted. This helped me understand why accuracy can be misleading when working with imbalanced healthcare datasets and why recall is especially important for this project.









Part F- Reflection

One thing I learned from this assignment is that high accuracy does not always mean a model is performing well. Both models achieved nearly 90% accuracy but struggled to identify patients who were actually readmitted within 30 days. This helped me understand the importance of choosing evaluation metrics based on the problem being addressed, especially when working with imbalanced healthcare data. In the future, I would explore adjusting the classification threshold or using class weights to improve recall and identify more patients at risk of readmission.

